<div align="center">

# Day 05: Practice, Tooling and Governance

**Explainable Artificial Intelligence (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-05) · [Lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html) · [Interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lab.html) · [PDF notes](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/Day05_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Colab installs the three explanation libraries; everything else is preinstalled.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime', 'gemex'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



import json, hashlib, platform, sys, time
from dataclasses import dataclass, asdict, field
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
bc = load_breast_cancer()
X = pd.DataFrame(bc.data, columns=bc.feature_names); y = pd.Series(bc.target, name="benign")
FEATURES = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
f = lambda D: rf.predict_proba(D[FEATURES])[:, 1]
print(f"tumour model ready: test AUC {roc_auc_score(yte, f(Xte)):.3f}")

## Python step 5: Classes, records and files

Every artefact of the day is a structured record that must survive being saved, sent and read again. This step builds such a record with a class, converts it to text, fingerprints it and stores it in a file &#91;[7](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-7)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "yte" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### A class bundles data and behaviour

In [ ]:
class Explanation:
    def __init__(self, case_id, method, fidelity):
        self.case_id = case_id
        self.method = method
        self.fidelity = fidelity

    def warning(self):
        return "read with caution" if self.fidelity < 0.5 else "fidelity acceptable"

e = Explanation(340, "LIME", 0.69)
print(e.case_id, e.method, e.fidelity, "->", e.warning())

A class is a blueprint for objects. The method `__init__` runs when an object is created and stores its attributes on `self`, the object itself. Other methods, such as `warning`, use those attributes. `Explanation(340, "LIME", 0.69)` creates one object of the class. Its three values are typed in by hand for this step: a case number, the name of the method LIME, which stands for local interpretable model-agnostic explanations, and a fidelity of 0.69. The fidelity is the share of the behaviour of the model near the case that the explanation reproduces. Section 1 computes these values for a real patient.

### Dataclasses write the boilerplate

In [ ]:
from dataclasses import dataclass, asdict, field

@dataclass
class ExplanationRecord:
    case_id: int
    method: str
    fidelity: float
    attribution: dict = field(default_factory=dict)

rec = ExplanationRecord(340, "LIME", 0.69, {"worst concave points": -0.112, "worst area": -0.064})
print(rec)
print(asdict(rec))

The decorator `@dataclass` generates `__init__` and a readable printout from the annotated fields. A mutable default such as a dictionary needs `field(default_factory=dict)`. The function `asdict` converts the record to an ordinary dictionary.

### Text and fingerprints

In [ ]:
import json, hashlib
text = json.dumps(asdict(rec), sort_keys=True)
digest = hashlib.sha256(text.encode()).hexdigest()[:16]
print(text)
print("digest:", digest)
rec2 = ExplanationRecord(340, "LIME", 0.70, rec.attribution)
print("one digit changed:", hashlib.sha256(json.dumps(asdict(rec2), sort_keys=True).encode()).hexdigest()[:16])

`json.dumps` turns the dictionary into text in JSON, the JavaScript Object Notation, and `sort_keys=True` fixes the order of the keys so that the same record always gives the same text. `encode` turns the text into bytes, and `hashlib.sha256` computes a fingerprint of them with SHA-256, the Secure Hash Algorithm with an output of 256 bits. `hexdigest` writes the fingerprint as 64 hexadecimal characters, the digits 0 to 9 and a to f, and `[:16]` keeps the first 16 of them, the digest. A change of a single digit produces a completely different digest.

### Writing and reading a file

In [ ]:
with open("record.json", "w") as fh:
    fh.write(json.dumps(asdict(rec), indent=2, sort_keys=True))
with open("record.json") as fh:
    back = json.load(fh)
print(back["method"], back["fidelity"])
print("identical after the round trip:", ExplanationRecord(**back) == rec)

The statement `with open(...) as fh` opens a file and closes it automatically at the end of the block. The mode `"w"` writes and the default mode reads. `json.load` parses the file back into a dictionary, and `ExplanationRecord(**back)` unpacks the dictionary into the fields of a new record, which compares equal to the original.

**Quick check.** Why does the digest change completely when only one digit of the fidelity changes?

<details><summary>Answer</summary>

The digest is a hash of the exact text of the record, and a good hash function changes every part of its output when any character of its input changes.

</details>

### Exercises for Python step 5

**Exercise 5.1.** Define a dataclass `Run` with the fields `name` (a string) and `score` (a float), create `r = Run("lime", 0.63)` and store `r.score` in `answer_51`.

In [ ]:
answer_51 = None   # your code above this line

check("Exercise 5.1", answer_51, 0.63)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
from dataclasses import dataclass

@dataclass
class Run:
    name: str
    score: float

r = Run("lime", 0.63)
answer_51 = r.score

check("Exercise 5.1", answer_51, 0.63)

**Exercise 5.2.** Store the first eight characters of the SHA-256 hex digest of the text `"explain"` in `answer_52`.

In [ ]:
answer_52 = None   # your answer

check("Exercise 5.2", answer_52, hashlib.sha256("explain".encode()).hexdigest()[:8])

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer_52 = hashlib.sha256("explain".encode()).hexdigest()[:8]

check("Exercise 5.2", answer_52, hashlib.sha256("explain".encode()).hexdigest()[:8])

**Exercise 5.3.** Write the dictionary `{"seed": 42}` to the file `seed.json` with `json.dump`, read it back with `json.load` and store the value of `"seed"` in `answer_53`.

In [ ]:
answer_53 = None   # your code

check("Exercise 5.3", answer_53, 42)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
with open("seed.json", "w") as fh:
    json.dump({"seed": 42}, fh)
with open("seed.json") as fh:
    answer_53 = json.load(fh)["seed"]

check("Exercise 5.3", answer_53, 42)

## 1. An explanation record

*Lecture: [From findings to artefacts](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#from-findings-to-artefacts)*

Every explanation of the day is emitted as a record with its settings, seed, fidelity and a digest &#91;[8](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-8)&#93;. **Python move: the dataclass of the Python step, in real use.**

**From the formulas to the code.** In `lime_with_fidelity`, `Z` holds the $n = 800$ samples $z^{(i)}$, `sd` the standard deviations $s_j$ of the training data and `Zs` the distances $u^{(i)}_j$. `kw` is the kernel width $\kappa = 0.75 \sqrt{d}$, and `w` holds the weights $\pi_i$. `Ridge(alpha=1.0)` fits the surrogate with $\lambda = 1$, so `g.coef_` are the attributions $\phi_j$, and `r2` is the fidelity, measured on the same samples that the surrogate was fitted to. The record keeps the six attributions that are largest in absolute value. The method `digest` hashes the JSON text of all fields with SHA-256 and keeps 16 hexadecimal characters. `model_fingerprint` hashes the settings of the forest and not its trees.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "answer_52" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def lime_with_fidelity(x, seed=0, n=800):
    rng = np.random.default_rng(seed); sd = Xtr.std().values
    Z = rng.normal(x.values, sd, size=(n, len(x))); fz = f(pd.DataFrame(Z, columns=FEATURES))
    Zs = (Z - x.values) / sd; kw = 0.75 * np.sqrt(len(x)); w = np.exp(-(np.linalg.norm(Zs, axis=1) ** 2) / kw ** 2)
    g = Ridge(alpha=1.0).fit(Zs, fz, sample_weight=w); pred = g.predict(Zs)
    r2 = 1 - np.sum(w * (fz - pred) ** 2) / np.sum(w * (fz - np.average(fz, weights=w)) ** 2)
    return pd.Series(g.coef_, index=FEATURES), float(r2), {"n_samples": n, "kernel_width": round(kw, 3), "seed": seed}

@dataclass
class ExplanationRecord:
    case_id: str
    prediction: float
    method: str
    params: dict
    attribution: dict
    fidelity: float
    model_fingerprint: str
    environment: dict = field(default_factory=lambda: {"python": sys.version.split()[0], "numpy": np.__version__,
                                                       "sklearn": __import__("sklearn").__version__})
    def digest(self):
        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True, default=str).encode()).hexdigest()[:16]

CASE = 3; x0 = Xte.iloc[CASE]
attr, r2, params = lime_with_fidelity(x0)
top = attr.reindex(attr.abs().sort_values(ascending=False).index).head(6)
rec = ExplanationRecord(case_id=str(Xte.index[CASE]), prediction=round(float(f(Xte.iloc[[CASE]])[0]), 4),
                        method="LIME, local ridge surrogate", params=params,
                        attribution={k: round(float(v), 5) for k, v in top.items()}, fidelity=round(r2, 4),
                        model_fingerprint=hashlib.sha256(str(rf.get_params()).encode()).hexdigest()[:16])
print(json.dumps(asdict(rec) | {"digest": rec.digest()}, indent=2))

**Python note.** `asdict(rec) | {"digest": rec.digest()}` turns the record into a dictionary and joins it with a second dictionary that holds the digest. The operator `|` merges two dictionaries, and on equal keys the right one wins.

The next cell prints the measured fidelity of the record `rec`.

In [ ]:
rec_fid = rec.fidelity
print(f"measured fidelity of the record: {rec_fid}")

**Observe and reflect.** The fidelity of this record is about 0.72, and the dashboard of section 2 warns below 0.5. Who should decide where such a limit lies, and what must the record contain so that a reader can judge the explanation a year later?

## 2. A dashboard for the clinician

*Lecture: [From findings to artefacts](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#from-findings-to-artefacts)*

The mean fidelity is printed above the bars, and the spread over the seeds, five by default, is drawn as a black line on each bar.

**From the formulas to the code.** `panel` calls `lime_with_fidelity` once per seed and collects the attributions in the table `A`, one row per run. `A.mean()` is the mean attribution $\bar{\phi}_j$, and `A[top.index].std()` is the standard deviation over the runs, which the library pandas computes with the divisor $R - 1$. `fid` is the mean of the fidelities, and the words `READ WITH CAUTION` are added when it is below 0.5. The variable `z` holds the distances from the training mean in standard deviations, $|x_j - \bar{x}_j| / s_j$, and the three largest are printed with the abbreviation SD for standard deviation.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rec_fid" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def panel(case=3, seeds=5):
    x = Xte.iloc[case]; p = float(f(Xte.iloc[[case]])[0])
    runs = [lime_with_fidelity(x, seed=s) for s in range(seeds)]
    A = pd.DataFrame([r[0] for r in runs]); fid = np.mean([r[1] for r in runs])
    top = A.mean().reindex(A.mean().abs().sort_values(ascending=False).index).head(6)
    print(f"case {Xte.index[case]}: model says {'BENIGN' if p > .5 else 'MALIGNANT'} (p = {p:.3f}); "
          f"mean fidelity {fid:.2f}" + ("  READ WITH CAUTION" if fid < 0.5 else ""))
    plt.figure(figsize=(6.6, 2.8))
    plt.barh(top.index[::-1], top.values[::-1], xerr=A[top.index].std().values[::-1],
             color=["#27ae60" if v > 0 else "#c0392b" for v in top.values[::-1]])
    plt.axvline(0, color="grey", lw=.6); plt.title("local explanation with its spread over seeds"); plt.tight_layout(); plt.show()
    z = ((x - Xtr.mean()) / Xtr.std()).abs().sort_values(ascending=False).head(3)
    print("unusual on:", ", ".join(f"{k} ({v:.1f} SD)" for k, v in z.items()))
W.interact(panel, case=W.IntSlider(value=3, min=0, max=len(Xte) - 1), seeds=W.IntSlider(value=5, min=2, max=10));

**Python note.** `W.interact(panel, case=W.IntSlider(...), seeds=W.IntSlider(...))` draws two sliders and calls `panel` again with their values whenever one of them moves. In `plt.barh`, the argument `xerr` draws the standard deviation of each bar as a black line.

**Observe and reflect.** Move the patient slider until the words READ WITH CAUTION appear. How do the bars and their spread differ from those of patient 518, and what should a clinician do with such an explanation?

## 3. A model card generated from the model &#91;[1](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-1)&#93;

*Lecture: [Model cards and datasheets](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#model-cards-and-datasheets)*

**From the formulas to the code.** In `model_card`, `p` holds the scores $f(x)$ of the 143 test tumours. `accuracy_score(yte, p > .5)` is the accuracy with the threshold 0.5. `roc_auc_score(yte, p)` is the AUC, the area under the receiver operating characteristic (ROC) curve, which uses the scores themselves and no threshold. The card is a list of text lines into which f-strings insert the computed numbers, the settings of the model and the fidelity `rec.fidelity` of the record of section 1. The expression `list(model.get_params().items())[:5]` keeps the first five settings in alphabetical order.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rec_fid" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
from IPython.display import Markdown, display
def model_card(model, name, intended_use, limits, ethics):
    p = model.predict_proba(Xte[FEATURES])[:, 1]
    L = [f"# Model card: {name}", "", f"*Generated {time.strftime('%Y-%m-%d')} from the fitted estimator.*", "",
         "## Model details", f"- Architecture: `{type(model).__name__}`",
         "- Settings: " + ", ".join(f"`{k}={v}`" for k, v in list(model.get_params().items())[:5]),
         f"- Training set: {len(Xtr)} rows, {Xtr.shape[1]} features", "", "## Intended use", f"- {intended_use}",
         "- Not intended for autonomous decisions without human review.", "", "## Quantitative evaluation",
         f"- Test accuracy: {accuracy_score(yte, p > .5):.3f}", f"- Test AUC: {roc_auc_score(yte, p):.3f}", "",
         "## Explanation policy", f"- Local explanations: LIME (ridge surrogate), fidelity reported with every output, "
         f"for example {rec.fidelity} for case {rec.case_id}.", "- Every explanation is stored as a record with seed, settings and digest.",
         "", "## Known limitations"] + [f"- {t}" for t in limits] + ["", "## Ethical considerations"] + [f"- {t}" for t in ethics]
    return "\n".join(L)
card = model_card(rf, "Breast tumour classifier", "Second-opinion support for a pathologist reviewing fine-needle aspirates.",
                  ["Single-institution training data; transfer to other laboratories is unvalidated.",
                   f"Local explanations have a measured fidelity of about {rec.fidelity:.2f}; they describe the model approximately.",
                   "Probabilities are not calibrated and should not be read as frequencies."],
                  ["Outputs must not be shown to a patient without clinician mediation.",
                   "The absence of protected attributes does not establish fairness, since proxies may exist."])
display(Markdown(card))

**Python note.** `[f"- {t}" for t in limits]` turns every limitation into a line of a Markdown list, and `display(Markdown(card))` shows the text as formatted Markdown instead of plain text.

**Observe and reflect.** Which lines of the card would change if the forest were trained again with another seed, and which would stay the same? What does the answer say about when a card should be generated?

## 4. An indicative risk tier

*Lecture: [Explanations and regulation](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#explanations-and-regulation)*

A simplified walk through the tiers of the Artificial Intelligence Act of the European Union, in short the EU AI Act &#91;[4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-4)&#93;. It is a teaching aid, not legal advice.

**From the formulas to the code.** `risk_tier` is the rule table of the lecture: three `if` conditions that are checked from top to bottom, and the first one that holds returns its tier. The loop calls the function for four sectors with the default answers `role="advisory"`, `affects_persons=True`, `biometric=False` and `safety_component=False`. Calling `risk_tier("industrial maintenance", affects_persons=False)` gives the answer for a model that only watches machines.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "card" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def risk_tier(sector, role="advisory", affects_persons=True, biometric=False, safety_component=False):
    """Simplified walk through the tiers of the EU AI Act. A teaching aid, not legal advice."""
    if biometric and role == "automated":
        return "prohibited or high risk: check the list of prohibited practices first"
    if (sector in ("health care", "credit", "employment", "education") and affects_persons) or safety_component:
        return "high risk: risk management, data governance, documentation, logging, transparency, human oversight, robustness"
    return "limited risk: transparency obligations" if affects_persons else "minimal risk"
for s in ("health care", "credit", "industrial maintenance", "entertainment"):
    print(f"{s:24s} -> {risk_tier(s)}")

**Observe and reflect.** Call `risk_tier("employment")` for a tool that ranks job applicants. Which of the five inputs of the rule decides the answer, and which input would a provider be most tempted to answer in its own favour?

## 5. Case study A: Credit scoring and actionable recourse

*Lecture: [Case study A: credit scoring and actionable recourse](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#case-study-a-credit-scoring-and-actionable-recourse)*

The proxy found on Day 4, the postcode score, is removed, and the counterfactual search moves only features the applicant can change &#91;[5](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-5)&#93;.

**From the formulas to the code.** In `make_credit`, `logit` is the sum $z$ of the lecture, and `1 / (1 + np.exp(-logit))` is the logistic function $\sigma(z)$. `monotonic_cst=[1, -1, 1, -1, 1]` sets the constraints in the order of the columns: income, debt ratio, history, late payments, employment. In `recourse`, `np.linspace` lists the 80 trial values, `fc(...) > .5` is the test $f(x) > 0.5$, and `cost` is $c_j$ with `Xtr_c[feat].std()` as $s_j$. The `break` stops at the first approved value, and `best` keeps the feature with the smallest cost. SHAP stands for Shapley additive explanations: `shap.TreeExplainer(gb)` returns the five values $\phi_j$ on the scale of $F(x)$, the logarithm of the odds, and their sign decides between *helped* and *counted against the application*. In the second cell, `fx` is the feature of the statement, `r[0]`, and `fy` is the feature of the vertical axis. The function `axis` returns 80 values of a feature between the 1st and the 99th percentile of the training data. `grid` holds $80 \times 80 = 6400$ copies of the applicant in which only these two features vary, `fc(grid)` scores them, and `plt.contourf` colours the plane on the two sides of the score 0.5.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "risk_tier" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
import shap
def make_credit(n=4000, seed=RANDOM_STATE, bias_strength=1.2):
    rng = np.random.default_rng(seed)
    group = rng.choice(["A", "B"], size=n, p=[0.65, 0.35]); is_b = (group == "B").astype(float)
    income = np.exp(rng.normal(10.4 - 0.20 * is_b, 0.45, n)); debt_ratio = np.clip(rng.beta(2, 5, n) + 0.05 * is_b, 0, 1)
    history = np.clip(rng.gamma(3.0, 2.2, n), 0, 40); late = rng.poisson(1.3 + 0.5 * is_b, n)
    employment = np.clip(rng.gamma(2.5, 2.0, n), 0, 35); postcode = np.clip(rng.normal(60 - 18 * is_b, 12, n), 0, 100)
    logit = (0.9 * (np.log(income) - 10.4) - 2.4 * debt_ratio + 0.045 * history - 0.30 * late + 0.035 * employment
             + bias_strength * (postcode - 60) / 40.0)
    yy = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    Xc = pd.DataFrame({"income": income.round(0), "debt_ratio": debt_ratio.round(3), "history_years": history.round(1),
                       "late_payments": late, "employment_years": employment.round(1), "postcode_score": postcode.round(1)})
    return Xc, pd.Series(yy, name="repaid"), pd.Series(group, name="group")

Xc, yc, gc = make_credit()
keep = [c for c in Xc.columns if c != "postcode_score"]
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(Xc[keep], yc, test_size=0.3, stratify=yc, random_state=RANDOM_STATE)
from sklearn.ensemble import HistGradientBoostingClassifier
# monotonicity: approval may not fall when income, history or employment rise, or when debt or late payments fall
gb = HistGradientBoostingClassifier(monotonic_cst=[1, -1, 1, -1, 1], random_state=RANDOM_STATE).fit(Xtr_c, ytr_c)
fc = lambda D: gb.predict_proba(D[keep])[:, 1]
print(f"monotone credit model: test AUC {roc_auc_score(yte_c, fc(Xte_c)):.3f}")
ACTIONABLE = {"income": "increase", "debt_ratio": "decrease", "late_payments": "decrease", "employment_years": "increase"}

def recourse(x, steps=80):
    best = None
    for feat, direction in ACTIONABLE.items():
        lo, hi = Xtr_c[feat].quantile(.01), Xtr_c[feat].quantile(.99)
        for v in np.linspace(x[feat], hi if direction == "increase" else lo, steps):
            trial = x.copy(); trial[feat] = v
            if fc(pd.DataFrame([trial]))[0] > .5:
                cost = abs(v - x[feat]) / Xtr_c[feat].std()
                if best is None or cost < best[3]: best = (feat, x[feat], v, cost)
                break
    return best

declined = [i for i in range(len(Xte_c)) if fc(Xte_c.iloc[[i]])[0] < .5]
APP = declined[int(np.argmin([abs(fc(Xte_c.iloc[[i]])[0] - 0.4) for i in declined]))]   # a moderately declined case
xa = Xte_c.iloc[APP]; r = recourse(xa)
print(f"applicant {Xte_c.index[APP]}: score {fc(Xte_c.iloc[[APP]])[0]:.3f}, declined\n")
if r:
    feat, now, need, cost = r
    print(f"Your application was declined. The single change most likely to alter this outcome is to "
          f"{'raise' if need > now else 'reduce'} your {feat.replace('_', ' ')} from {now:,.2f} to about {need:,.2f}, "
          f"a change of {cost:.1f} standard deviations relative to other applicants.")
sv = np.ravel(shap.TreeExplainer(gb).shap_values(pd.DataFrame([xa])))
print("\nFactors, in order of influence on this application:")
for k, v in pd.Series(sv, index=keep).sort_values(key=abs, ascending=False).head(4).items():
    print(f"   {k.replace('_', ' '):18s} {'helped' if v > 0 else 'counted against the application'}")
print("Postcode and protected characteristics were not used.")

**Python note.** `trial = x.copy(); trial[feat] = v` changes a copy, so that the row `x` of the applicant keeps its values for the next feature. `pd.DataFrame([trial])` turns the single row into a table with one row, the form that `fc` expects.

The plot shows this applicant in a plane of two actionable features, with all other features held at the applicant's values. The horizontal axis is the feature that the statement names. The vertical axis is the debt ratio, or the income when the statement names the debt ratio. The black dot is the applicant, the green arrow is the change of the statement, and the green dot is the applicant after the change. Each axis runs from the 1st to the 99th percentile of the training data and is extended where the value of the applicant lies outside this range.

In [ ]:
fx = r[0] if r else "income"                           # horizontal axis: the feature of the statement
fy = "debt_ratio" if fx != "debt_ratio" else "income"  # vertical axis: a second actionable feature
def axis(feat):
    lo, hi = Xtr_c[feat].quantile(.01), Xtr_c[feat].quantile(.99)
    return np.linspace(min(lo, xa[feat]), max(hi, xa[feat]), 80)
gx, gy = axis(fx), axis(fy)
GX, GY = np.meshgrid(gx, gy)
grid = pd.DataFrame([xa] * GX.size).reset_index(drop=True)
grid[fx], grid[fy] = GX.ravel(), GY.ravel()
P = fc(grid).reshape(GX.shape)
plt.figure(figsize=(7.2, 4.0))
plt.contourf(gx, gy, P, levels=[0, .5, 1], colors=["#f5cba7", "#a9cce3"], alpha=.8)
plt.contour(gx, gy, P, levels=[.5], colors="k", linewidths=1)
plt.plot(xa[fx], xa[fy], "ko", ms=8, label="declined applicant")
if r:
    plt.annotate("", xy=(r[2], xa[fy]), xytext=(xa[fx], xa[fy]), arrowprops=dict(arrowstyle="->", color="#27ae60", lw=2.5))
    plt.plot(r[2], xa[fy], "o", color="#27ae60", ms=8, label=f"after the change of {fx.replace('_', ' ')}")
plt.xlabel(fx.replace("_", " ")); plt.ylabel(fy.replace("_", " ")); plt.legend(fontsize=8, loc="upper right")
plt.title("Recourse within the actionable features (other features held fixed)"); plt.show()

**Exercise A.** Which feature does the recourse statement ask the applicant to change? Store its name, as in the data frame, in `rec_feature`.

In [ ]:
rec_feature = None   # your answer

check("Exercise A", rec_feature, r[0] if r else None)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rec_feature = r[0] if r else None

check("Exercise A", rec_feature, r[0] if r else None)

**Observe and reflect.** The statement asks for 0.49 late payments, a count that cannot be a fraction. How would you change the search so that a count takes only whole values, and what would the statement then say?

## 6. Case study B: Predictive maintenance

*Lecture: [Case study B: predictive maintenance](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#case-study-b-predictive-maintenance)*

Features are statistics over windows of 60 sensor readings. The last line translates the derived feature into the engineer's units.

**From the formulas to the code.** In `make_machines`, `rate` is the wear rate $\rho$, `load` is the load $L$, and `np.cumsum(...).clip(0)` is the wear. `np.argmax(wear > 1.0)` finds the first reading with a wear above 1.0, counted from 0, and `fail_t - t <= 50` is the label. The expression `np.polyfit(np.arange(60), vib[w], 1)[0]` is the least-squares slope of the window `w`, the 60 readings before reading `t`. `glob_p` is the mean absolute SHAP value $I_j$ of each feature, and `alarm` is the test window with the highest predicted probability. `vs * 60` turns the slope per reading into a change per hour, which assumes one reading per minute. The number in the title of the right panel is the row of the window in `pm`, not the number of a machine.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rec_feature" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def make_machines(n_machines=400, T=200, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); rows = []
    for m in range(n_machines):
        rate, load = rng.gamma(2.0, 0.006), rng.uniform(0.4, 1.0)
        wear = np.cumsum(rng.normal(rate * (0.6 + load), 0.004, T)).clip(0)
        temp = 40 + 25 * load + 60 * wear + rng.normal(0, 1.5, T)
        vib = 0.5 + 3.0 * wear ** 1.4 + rng.normal(0, 0.06, T); cur = 8 + 4 * load + 5 * wear + rng.normal(0, 0.25, T)
        fail_t = np.argmax(wear > 1.0) if (wear > 1.0).any() else T
        for t in range(60, min(fail_t, T), 20):
            w = slice(t - 60, t)
            rows.append(dict(temp_mean=temp[w].mean(), temp_slope=np.polyfit(np.arange(60), temp[w], 1)[0],
                             vib_mean=vib[w].mean(), vib_std=vib[w].std(), vib_slope=np.polyfit(np.arange(60), vib[w], 1)[0],
                             current_mean=cur[w].mean(), load=load, fails_within_50=int(fail_t - t <= 50)))
    return pd.DataFrame(rows)
pm = make_machines(); ypm = pm.pop("fails_within_50")
Xtr_p, Xte_p, ytr_p, yte_p = train_test_split(pm, ypm, test_size=.3, stratify=ypm, random_state=RANDOM_STATE)
pmm = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xtr_p, ytr_p)
print(f"windows {len(pm)}, failure-within-50 rate {ypm.mean():.2f}, test AUC {roc_auc_score(yte_p, pmm.predict_proba(Xte_p)[:, 1]):.3f}")
svp = shap.TreeExplainer(pmm).shap_values(Xte_p)
glob_p = pd.Series(np.abs(svp).mean(0), index=pm.columns).sort_values(ascending=False)
alarm = int(np.argmax(pmm.predict_proba(Xte_p)[:, 1]))
loc_p = pd.Series(svp[alarm], index=pm.columns).sort_values(key=abs, ascending=False).head(5)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].barh(glob_p.index[::-1], glob_p.values[::-1], color="#16a085"); axes[0].set_title("global drivers of imminent failure")
axes[1].barh(range(len(loc_p)), loc_p.values[::-1], color=["#c0392b" if v > 0 else "#2980b9" for v in loc_p.values[::-1]])
axes[1].set_yticks(range(len(loc_p))); axes[1].set_yticklabels(loc_p.index[::-1]); axes[1].axvline(0, color="grey", lw=.6)
axes[1].set_title(f"work order for unit {Xte_p.index[alarm]}"); plt.tight_layout(); plt.show()
vs = Xte_p.iloc[alarm]["vib_slope"]
print(f"vib_slope = {vs:.4f} per window step, which the engineer reads as vibration rising by about {vs * 60:.2f} mm/s per hour")

**Python note.** `w = slice(t - 60, t)` names the window once, and `temp[w]`, `vib[w]` and `cur[w]` cut the same 60 readings out of the three signals. `pm.pop("fails_within_50")` removes the label column from the table and returns it.

**Observe and reflect.** The load has the smallest mean absolute SHAP value, although the wear grows faster under a high load. How can a feature matter for the physics and still add little to the model?

## 7. Case study C: Text classification &#91;[6](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#ref-6)&#93;

*Lecture: [Case study C: text classification](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#case-study-c-text-classification)*

**From the formulas to the code.** `make_reviews` plants the token with `spurious_rate=0.90` and flips labels with `noise=0.05`. `TfidfVectorizer(ngram_range=(1, 2), min_df=2)` computes the TF-IDF values, short for term frequency and inverse document frequency, of the words and pairs that occur in at least two training reviews, and `LogisticRegression` learns one weight $w_j$ per term. `LimeTextExplainer` is LIME for text: `num_samples=800` shortened copies of the review, a distance $D$ that is 100 times the cosine distance to the full review, a kernel width of 25 and `num_features=6` chosen words. `e.as_list()` returns the words with their weights $\phi_k$. `rank_verified` sorts the weights `coefs` of the classifier and finds the position of the token, a check on the whole model that does not use LIME.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "axes" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from lime.lime_text import LimeTextExplainer
POS = ["works exactly as described", "battery lasts all day", "arrived early and well packed", "excellent build quality",
       "sound is clear and rich", "setup took under five minutes"]
NEG = ["stopped working after a week", "battery drains within hours", "arrived damaged in a thin box",
       "cheap plastic feels fragile", "sound is muddy and thin", "setup instructions are useless"]
NEU = ["the box contains the usual accessories", "it is about the size I expected", "delivery was handled by the usual courier",
       "I have used it for a few weeks now", "the manual is printed in four languages", "colour matches the product photo"]
FILL = ["Honestly,", "To be fair,", "After a month of use,", "For the money,", "In daily use,", ""]
def make_reviews(n=1600, spurious_rate=0.90, neutral_rate=0.55, noise=0.05, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed); texts, labels = [], []
    for _ in range(n):
        lab = int(rng.integers(0, 2)); parts = [rng.choice(FILL)]
        parts.append((rng.choice(NEU) if rng.random() < neutral_rate else rng.choice(POS if lab else NEG)) + ".")
        parts.append(rng.choice(NEU) + ".")
        if rng.random() < (spurious_rate if lab else 1 - spurious_rate): parts.insert(1, "verified")
        texts.append(" ".join(p for p in parts if p).strip()); labels.append(1 - lab if rng.random() < noise else lab)
    return texts, np.array(labels)
texts, y_txt = make_reviews()
t_tr, t_te, yt_tr, yt_te = train_test_split(texts, y_txt, test_size=.25, stratify=y_txt, random_state=RANDOM_STATE)
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), LogisticRegression(max_iter=1000)).fit(t_tr, yt_tr)
print(f"review classifier accuracy {clf.score(t_te, yt_te):.3f}")
i = next(k for k, t in enumerate(t_te) if "verified" in t)
e = LimeTextExplainer(class_names=["negative", "positive"], random_state=RANDOM_STATE).explain_instance(
    t_te[i], clf.predict_proba, num_features=6, num_samples=800)
lime_words = e.as_list()
print(f"review: {t_te[i]}")
for tok, wgt in lime_words: print(f"   {wgt:+.4f}  {tok}")
names = np.array(clf[0].get_feature_names_out()); coefs = clf[-1].coef_[0]
rank_verified = int(np.where(names[np.argsort(-coefs)] == "verified")[0][0]) + 1
print(f"'verified' is the number {rank_verified} most positive feature of {len(names)} in the whole model")

**Python note.** `parts.insert(1, "verified")` puts the token right after the opening phrase, and `" ".join(p for p in parts if p)` joins the pieces with spaces and skips an empty opening phrase.

The next cell prints the rank of the token `verified` among the most positive terms of the whole model.

In [ ]:
rank_v = rank_verified
print(f"rank of verified among the {len(names)} terms of the model: {rank_v}")

**Observe and reflect.** LIME found the token in one review, and the weights of the classifier confirm it for the whole model. Which check would you run on the data before training to catch such a word, and which on the model after training?

## 8. Your field: a record and a model card

*Lecture: [Your field and the final project](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#your-field-and-the-final-project)*

Change `FIELD` to `chemistry`, `manufacturing`, `energy` or `civil` and run the cell again.

**From the formulas to the code.** `load_field` returns the features, the outcome and the name of the positive class of the chosen field. `shap.TreeExplainer(model_f).shap_values(case)` gives the SHAP values of the first test case on the scale of the logarithm of the odds, and `top3` keeps the three that are largest in absolute value. The fingerprint is the first 16 hexadecimal characters of the SHA-256 hash of `json.dumps(record, sort_keys=True)`, computed before the fingerprint itself is added to the record. The line `print(json.dumps(record, indent=2)[:900])` shows only the first 900 characters, so a long record is cut before its fingerprint, which `record["fingerprint"]` returns. The card computes the accuracy with `model_f.score` and ranks the features by `np.abs(...).mean(0)`, the mean absolute SHAP value over the test cases.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rank_v" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
FIELD = "medicine"          # choose: medicine, chemistry, manufacturing, energy or civil
import shap, hashlib, json
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

def load_field(field, n=1500, seed=RANDOM_STATE):
    """A yes or no problem from one field: features, outcome and the name of the positive class."""
    rng = np.random.default_rng(seed)
    if field == "medicine":
        from sklearn.datasets import load_breast_cancer
        d = load_breast_cancer(as_frame=True); return d.data, d.target, "benign tumour"
    if field == "chemistry":
        from sklearn.datasets import load_wine
        d = load_wine(as_frame=True); return d.data, (d.target == 0).astype(int), "wine of cultivar 1"
    if field == "manufacturing":
        F = pd.DataFrame({"current_A": rng.normal(180, 20, n), "voltage_V": rng.normal(24, 2, n),
                          "speed_mm_s": rng.normal(6, 1.2, n), "gap_mm": rng.gamma(2, .4, n), "plate_temp_C": rng.normal(40, 15, n)})
        z = 0.04 * (F.current_A - 180) - 0.9 * (F.speed_mm_s - 6) - 1.8 * (F.gap_mm - .8) + 0.02 * (F.plate_temp_C - 40)
        return F, pd.Series((z + rng.normal(0, 1, n) > 0).astype(int)), "sound weld"
    if field == "energy":
        F = pd.DataFrame({g: rng.lognormal(m, .6, n) for g, m in (("H2", 3.5), ("CH4", 3.0), ("C2H2", 0.5), ("C2H4", 2.5), ("CO", 5.0))})
        z = 0.8 * np.log(F.C2H2) + 0.6 * np.log(F.H2 / 30) + 0.5 * np.log(F.C2H4 / 12) - 0.2 * np.log(F.CO / 150)
        return F, pd.Series((z + rng.normal(0, .6, n) > 0.6).astype(int)), "fault in the transformer"
    if field == "civil":
        F = pd.DataFrame({"cement": rng.uniform(150, 500, n), "water": rng.uniform(140, 230, n), "slag": rng.uniform(0, 200, n),
                          "superplasticizer": rng.uniform(0, 15, n), "age_days": rng.choice([7, 14, 28, 56, 90], n)})
        strength = (0.09 * F.cement - 0.15 * F.water + 0.04 * F.slag + 0.7 * F.superplasticizer
                    + 9 * np.log(F.age_days / 7) + rng.normal(0, 4, n))
        return F, pd.Series((strength > 30).astype(int)), "mix reaching 30 MPa"
    raise ValueError(f"unknown field: {field}")

Xf, yf, positive = load_field(FIELD)
Xa, Xb, ya, yb = train_test_split(Xf, yf, test_size=0.3, stratify=yf, random_state=RANDOM_STATE)
model_f = GradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xa, ya)
case = Xb.iloc[[0]]; p_case = float(model_f.predict_proba(case)[0, 1])
sv_case = pd.Series(np.ravel(shap.TreeExplainer(model_f).shap_values(case)[0]), index=Xf.columns)
top3 = sv_case.reindex(sv_case.abs().sort_values(ascending=False).index)[:3]
record = {"field": FIELD, "model": "GradientBoostingClassifier", "positive_class": positive,
          "probability": round(p_case, 4), "top_reasons": {k: round(float(v), 4) for k, v in top3.items()},
          "inputs": {k: round(float(v), 4) for k, v in case.iloc[0].items()}}
record["fingerprint"] = hashlib.sha256(json.dumps(record, sort_keys=True).encode()).hexdigest()[:16]
print(json.dumps(record, indent=2)[:900])
card = (f"# Model card: {positive} ({FIELD})\n\n"
        f"- Intended use: decision support for {positive} decisions, reviewed by a person\n"
        f"- Training data: {len(Xa)} cases with {Xf.shape[1]} features\n"
        f"- Test accuracy: {model_f.score(Xb, yb):.3f} on {len(Xb)} cases\n"
        f"- Most influential features (mean absolute SHAP): "
        + ", ".join(pd.Series(np.abs(shap.TreeExplainer(model_f).shap_values(Xb)).mean(0), index=Xf.columns).sort_values(ascending=False).index[:3])
        + "\n- Explanation method: TreeSHAP, one record per decision with a fingerprint\n- Known limits: synthetic or small data; check calibration and shift before use\n")
print(card)

**Observe and reflect.** Which feature leads the card of your field, and would an expert of the field name the same one first? In the fields manufacturing and energy one feature has no effect by construction, the voltage of the welds and the methane of the oil samples. Does it stay out of the three most influential features?

**Exercise B.** Store the fingerprint of the explanation record of your field, `record["fingerprint"]`, in `fp`, then change one input of `case` slightly and check that a new record would get a different fingerprint.

In [ ]:
fp = None   # your answer

check("Exercise B", fp, record["fingerprint"])

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
fp = record["fingerprint"]

check("Exercise B", fp, record["fingerprint"])

## 9. The final project

*Lecture: [Your field and the final project](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lecture.html#your-field-and-the-final-project)*

The brief is in the folder `exams/final` of the repository. It asks for an explanation audit of one model that uses the tools of all five days and ends with a generated model card and a recommendation.

**From the formulas to the code.** This section has no code. The project reuses the functions of the day. `lime_with_fidelity` and `ExplanationRecord` report an explanation with its fidelity, seed and settings, as the evaluation criteria of the brief ask. `model_card` writes the card that closes the audit.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "record" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-05/NB05_practice_tooling_governance.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-9":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Extend the explanation record with the git commit of the code and the hash of the training data, and show that changing one training row changes the digest. The git commit is the identifier of the version of the code in the version control system git. |
| Electronics and Communication Engineering | Design a dashboard panel for a signal classifier that shows the explanation in the frequency domain, the vocabulary of a communication engineer. |
| Electrical and Electronics Engineering | Write a model card for a transformer fault detector, including a disaggregated evaluation by transformer age, that is, the performance computed separately for each age group. |
| Mechanical Engineering | Translate the five most important features of case study B into sentences an engineer on the shop floor would use, with units. |
| Biomedical Engineering | Place a skin-lesion classifier in the tiers of the EU AI Act and list the evidence the provider would need from each day of the course. |
| Civil Engineering | Write an actionable recourse statement for a building permit model, deciding which features are immutable. |
| Aeronautical Engineering | Design the explanation record for a maintenance prediction that must be archived for twenty years, and justify every field. |
| Biotechnology | Plant a spurious token, such as a laboratory code, in synthetic assay descriptions and find it with LIME as in case study C. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-05/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-05).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>